# 2.RunnableParallel模拟检索

知识点讲解：用 RunnableParallel 构建 RAG 输入（隐式字典 + itemgetter）

1. 本示例解决的核心问题
   - RAG（检索增强生成）的 Prompt 通常需要两个变量：context（检索到的资料）与 query（用户问题）
   - 但调用方只想传一个 {"query": "..."}，不希望手动准备 context
   - 解决方案：在链最前面放一个「并行字典」，一个分支去检索生成 context，
     另一个分支把原始 query 透传出来，两者汇总成 prompt 需要的完整输入

2. RunnableParallel 的职责定位（回顾）
   - 官方定义：用于「操作 Runnable 的输出，以匹配序列中下一个 Runnable 的输入」，
     并行运行多个 Runnable 并格式化输出结构
   - 本示例正是用到它的「格式化输出结构」职责：把上游单一输入整形成下游 prompt 需要的两字段字典
     （「并行执行」职责见 1.RunnableParallel使用技巧.py）

3. LCEL 中 dict 的隐式转换规则（关键知识点）
   - 在管道中直接写 dict，__or__ / __ror__ 会通过 coerce_to_runnable 自动转成 RunnableParallel
   - dict 的 value 若是普通函数 / lambda，会被自动包装成 RunnableLambda；
     value 若是 dict，会递归包装成嵌套的 RunnableParallel
   - 因此 {"context": lambda x: ..., "query": itemgetter("query")} 完全等价于
     RunnableParallel(context=RunnableLambda(...), query=RunnableLambda(itemgetter("query")))

4. 等价写法：显式 RunnableParallel
   - 课程文档给出的同语义写法是显式传参，value 甚至可以直接传普通函数（由 coerce_to_runnable 包装）：
       chain = RunnableParallel(
           context=retrieval,           # retrieval 是普通函数，自动包成 RunnableLambda
           query=RunnablePassthrough(), # 透传整个输入
       ) | prompt | llm | parser
   - 两个分支并行执行：retrieval(x["query"]) 生成 context，RunnablePassthrough() 把原始输入整体作为 query
   - 与隐式 dict 写法输出一致，区别在于显式写法更直观、隐式写法更简洁

5. operator.itemgetter 的作用与优势
   - itemgetter("query") 返回一个可调用对象，效果等同 lambda x: x["query"]
   - 优势：C 实现，性能更好；语义明确，专用于「从 dict 取字段」
   - 课程文档提示：若输入是字典且只想取其中一部分，itemgetter 是标准做法
   - 在 LCEL 中极为常见，是从上游字典提取单个字段的惯用写法

6. lambda 在 LCEL 中的自动包装
   - lambda x: retrieval(x["query"]) 会被包装成 RunnableLambda
   - RunnableLambda 让任意 Python 函数获得 Runnable 能力（invoke / batch / stream）
   - 函数只接受「一个参数」，即上游传来的完整输入；多参数需借助 functools.partial 或 bind

7. 数据流转全过程
   {"query": "你好，我是谁?"}
        ├─→ lambda x: retrieval(x["query"]) ──→ "我是慕小课"        → context
        └─→ itemgetter("query")            ──→ "你好，我是谁?"      → query
                                    ↓
        {"context": "我是慕小课", "query": "你好，我是谁?"}
                                    ↓ prompt
        [HumanMessage(填充好 context 与 query 的完整提示词)]
                                    ↓ llm
        AIMessage(content="...")
                                    ↓ parser
        "最终回答字符串"

8. 与 RunnablePassthrough.assign 的对比（见本章第 3 个示例）
   - 本写法：必须「显式列出」prompt 需要的每一个字段，漏写会报 KeyError
   - assign 写法：自动保留原输入的所有字段，只需追加新增的 context 字段，更简洁
   - 字段多时推荐 assign；需要完全重塑输入结构时用显式 dict 更清晰

9. 注意事项
   - 字典分支是并行执行的，retrieval 与 itemgetter 同时跑（itemgetter 极快，感知不到）
   - 检索函数若有副作用（打印日志、写库），并行下的执行顺序不保证
   - 真实 RAG 中 retrieval 通常返回 List[Document]，
     需再拼接成字符串（"\n\n".join(doc.page_content for doc in docs)）后填入 context


In [ ]:
from operator import itemgetter
import dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()

In [ ]:
def retrieval(query: str) -> str:
    """
    一个模拟的检索器函数
    作用：模拟真实 RAG 中的向量检索环节，根据用户问题返回相关上下文
    参数：
        query: str - 用户的原始提问文本
    返回：
        str - 检索到的上下文文本（这里固定返回一段模拟资料）
    真实场景对照：
        真实实现通常是 vector_store.similarity_search(query, k=4)，
        返回 List[Document] 后再用 "\\n\\n".join(doc.page_content for doc in docs) 拼接
    """
    print("正在检索:", query)
    return "我是慕小课"

In [ ]:
prompt = ChatPromptTemplate.from_template("""请根据用户的问题回答，可以参考对应的上下文进行生成。
<context>
{context}
</context>
用户的提问是: {query}""")
llm = ChatOpenAI(model="deepseek-flash")
parser = StrOutputParser()
chain = {
            "context": lambda x: retrieval(x["query"]),
            "query": itemgetter("query"),
        } | prompt | llm | parser
content = chain.invoke({"query": "你好，我是谁?"})
print(content)